In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.gold.dim_categoria_ruralidad")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.dim_categoria_ruralidad (
  categoria_ruralidad_id INT,
  categoria_ruralidad_descripcion STRING
)
USING DELTA
""")

In [0]:
import pandas as pd

df = spark.table("workspace.silver.predios_registro").toPandas()

In [0]:
from pyspark.sql import functions as F

dim_rural = df[[
    "categoria_ruralidad_descripcion"
]].dropna().drop_duplicates().reset_index(drop=True)

dim_rural["categoria_ruralidad_id"] = dim_rural.index + 1

dim_rural = dim_rural[[
    "categoria_ruralidad_id",
    "categoria_ruralidad_descripcion"
]]

df_spark = spark.createDataFrame(dim_rural)

df_spark = df_spark.withColumn("categoria_ruralidad_id", F.col("categoria_ruralidad_id").cast("int"))

In [0]:

# Usamos overwrite para reemplazar completamente los datos existentes
df_spark.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.dim_categoria_ruralidad")